In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import stackview

from scipy.ndimage import zoom

In [43]:
# Find files
exps = 'exp_197'
scan = 'Scan_7'

# Create paths
scansFolder = r'C:\Users\Lab User\Desktop\ModernExperiments'
file = os.path.join(scansFolder,exps,scan+'.hdf5')
dataset = r'/RawData/' + scan

In [44]:
# Import data
with h5py.File(file,'r') as f: 
    data = f[dataset][()] # (slices, rows, width)
    sz = data.shape
img = data[data.shape[0]//2,:,:]

In [45]:
plt.imshow(img, cmap='gray')
plt.title("Click 4 points, then Enter")
coords = plt.ginput(4, timeout=0)  # timeout=0 means wait indefinitely
plt.show()

In [46]:
xs = [int(c[0]) for c in coords]
ys = [int(c[1]) for c in coords]

col_min, col_max = min(xs), max(xs)
row_min, row_max = min(ys), max(ys)

croppedData = data[:, row_min:row_max, col_min:col_max]
properData  = np.transpose(croppedData,[1,2,0]) # (rows, width, slices)

In [47]:
sPath = r'C:\Users\Lab User\Desktop\temp1\Granular-Compression\Data'
sName = r'\p'+scan+'.hdf5'
sFull = sPath+sName

In [36]:
flatData = np.concatenate([np.ravel(slice_) for slice_ in properData])
with h5py.File(sFull, 'w') as f: 
    f.create_dataset("default", data=flatData)
    f.attrs["shape"] = properData.shape

In [48]:
rescaleData = zoom(properData,0.25)
stackview.orthogonal(rescaleData, continuous_update=True)